# Notebook 2: Biomedical Discriminative Encoders (PubMedBERT & BioLinkBERT)
**Thesis:** Rhetorical Role Classification in Medical Abstracts (Sequential Text Tagging)  
**Field:** Biomedical Natural Language Processing (BioNLP) / Biomedical AI  
**Compute Environment:** Kaggle Notebook (GPU T4 x2, fp16 mixed precision, PyTorch DataParallel)  
**Target Classes (5):** `BACKGROUND`, `OBJECTIVE`, `METHODS`, `RESULTS`, `CONCLUSION`  

---

### Objectives of this Notebook:
1. **Model Architectures Evaluated:**
   - Primary: `microsoft/BiomedNLP-PubMedBERT-base-uncased-abstract` (Model B from thesis plan)
   - Secondary: `michiyasunaga/BioLinkBERT-base` (Biomedical knowledge-graph augmented encoder)
   - Optional configurable alternatives: `allenai/scibert_scivocab_uncased`, `dmis-lab/biobert-v1.1`
2. **Context & Positional Fusion:** Incorporate preceding/succeeding sentence discourse context plus continuous normalized position features (`normalized_pos`, `line_number`, `total_lines`) via a dual-stream multimodal classification head.
3. **Multi-GPU & Precision:** Leverage dual NVIDIA T4 GPUs (`T4 x2`) via PyTorch `DataParallel` and mixed-precision `fp16` (no `bf16`, which T4 hardware does not support).
4. **Class Imbalance Handling:** Support Standard Cross-Entropy, Class-Weighted Cross-Entropy, and Focal Loss ($\gamma=2.0$).
5. **Multi-Dataset Training:** Train and evaluate on:
   - Dataset 1: 5k Medical Abstracts (`medical_abstracts_5k`)
   - Dataset 2: `PubMed_20k_RCT`
   - Dataset 3: `PubMed_200k_RCT` (with pre-flight benchmark and configurable subset size)
6. **Evaluation Framework (Phase 4):** Compute Accuracy, Macro-F1, Micro-F1, Per-class Precision/Recall/F1, 5x5 Confusion Matrix (PNG and CSV), and Sequential Transition Consistency ($P(y_t \mid y_{t-1})$).
7. **Artifact Export & Git Push:** Save best models, metrics, and push incrementally to GitHub via Kaggle Secrets with $<100$ MB file limit checks.

### Step 1: Environment Verification & Dependency Installation
Check library versions (`transformers`, `torch`, `scikit-learn`, `matplotlib`, `seaborn`) and confirm dependencies are present.

In [1]:
import os
import sys
import subprocess

# Ensure essential dependencies are installed
required_pkgs = ["transformers", "accelerate", "datasets", "evaluate", "scikit-learn", "seaborn", "pyarrow"]
missing = []
for pkg in required_pkgs:
    try:
        __import__(pkg)
    except ImportError:
        missing.append(pkg)

if missing:
    print(f"Installing missing packages: {missing}")
    subprocess.check_call([sys.executable, "-m", "pip", "install", "-q"] + missing)

from typing import Dict, List, Tuple, Any, Optional
import torch
import torch.nn as nn
import torch.nn.functional as F
from torch.utils.data import Dataset, DataLoader
import transformers
from transformers import AutoTokenizer, AutoModel, AutoConfig, get_cosine_schedule_with_warmup
import sklearn
from sklearn.metrics import classification_report, confusion_matrix, f1_score, accuracy_score, precision_recall_fscore_support
import matplotlib.pyplot as plt
import seaborn as sns
import numpy as np
import pandas as pd

print("=" * 80)
print("DEPENDENCY & VERSION AUDIT")
print("=" * 80)
print(f"PyTorch Version:      {torch.__version__}")
print(f"Transformers Version: {transformers.__version__}")
print(f"Scikit-Learn Version: {sklearn.__version__}")
print(f"CUDA Available:       {torch.cuda.is_available()}")
if torch.cuda.is_available():
    print(f"Device Count:         {torch.cuda.device_count()}")
    for i in range(torch.cuda.device_count()):
        print(f"  GPU {i}: {torch.cuda.get_device_name(i)} | Capability: {torch.cuda.get_device_capability(i)} | VRAM: {torch.cuda.get_device_properties(i).total_memory / 1e9:.2f} GB")
print("=" * 80)

Installing missing packages: ['evaluate', 'scikit-learn']
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 84.1/84.1 kB 2.2 MB/s eta 0:00:00
DEPENDENCY & VERSION AUDIT
PyTorch Version:      2.11.0+cu128
Transformers Version: 5.16.1
Scikit-Learn Version: 1.6.1
CUDA Available:       True
Device Count:         2
  GPU 0: Tesla T4 | Capability: (7, 5) | VRAM: 15.64 GB
  GPU 1: Tesla T4 | Capability: (7, 5) | VRAM: 15.64 GB


### Step 2: Master Configuration Cell
Centralized configuration controlling data inputs, model selection, training hyperparameters, loss functions, and hardware settings.

In [2]:
# Master Hyperparameter & Hardware Configuration
RANDOM_SEED = 42
torch.manual_seed(RANDOM_SEED)
np.random.seed(RANDOM_SEED)
if torch.cuda.is_available():
    torch.cuda.manual_seed_all(RANDOM_SEED)

# Data Locations (Checks Kaggle dataset mount first, then local fallback)
def resolve_data_path(candidate_paths):
    for p in candidate_paths:
        if os.path.exists(p):
            return p
    return None

DATA_DIR = resolve_data_path([
    "/kaggle/input/datasets/siamabir/rhetorical-roles-processed-data",
    "/kaggle/input/rhetorical-roles-processed-data",
    "/kaggle/input/datasets/siamabir/rhetorical-role-classification-in-medical-abstract/processed_data",
    "/kaggle/input/rhetorical-role-classification-in-medical-abstract/processed_data",
    "/kaggle/working/processed_data",
    "./processed_data",
    "../processed_data"
])

# Auto-resolve nested directory (handles datasets packaged with a subfolder)
if DATA_DIR and os.path.exists(DATA_DIR):
    has_direct_files = any(f.endswith(".parquet") for f in os.listdir(DATA_DIR) if os.path.isfile(os.path.join(DATA_DIR, f)))
    if not has_direct_files:
        for entry in os.listdir(DATA_DIR):
            cand = os.path.join(DATA_DIR, entry)
            if os.path.isdir(cand) and any(f.endswith(".parquet") for f in os.listdir(cand) if os.path.isfile(os.path.join(cand, f))):
                DATA_DIR = cand
                break

print(f"Using processed data directory: {DATA_DIR}")

def find_processed_file(fname: str, base_dir: str) -> str:
    """Finds a file whether directly in base_dir or nested in a subfolder."""
    if base_dir and os.path.exists(base_dir):
        p = os.path.join(base_dir, fname)
        if os.path.exists(p):
            return p
        for root, _, files in os.walk(base_dir):
            if fname in files:
                return os.path.join(root, fname)
    raise FileNotFoundError(f"File '{fname}' could not be found inside: {base_dir}")

# Models to Train & Compare
# Verified against Hugging Face Hub API:
# 1. microsoft/BiomedNLP-PubMedBERT-base-uncased-abstract (Plan Model B)
# 2. michiyasunaga/BioLinkBERT-base (Biomedical relation & discourse benchmark)
MODELS_TO_TRAIN = [
    {
        "model_id": "microsoft/BiomedNLP-PubMedBERT-base-uncased-abstract",
        "name": "PubMedBERT-base"
    },
    {
        "model_id": "michiyasunaga/BioLinkBERT-base",
        "name": "BioLinkBERT-base"
    }
]

# Target Classes
TARGET_CLASSES = ["BACKGROUND", "OBJECTIVE", "METHODS", "RESULTS", "CONCLUSION"]
NUM_CLASSES = len(TARGET_CLASSES)
LABEL2ID = {lbl: idx for idx, lbl in enumerate(TARGET_CLASSES)}
ID2LABEL = {idx: lbl for idx, lbl in enumerate(TARGET_CLASSES)}

# Training Hyperparameters
MAX_SEQ_LENGTH = 256        # Target + Context tokens
BATCH_SIZE_PER_GPU = 32     # Effective batch size = 32 * num_gpus = 64 on T4 x2
LEARNING_RATE = 3e-5
WEIGHT_DECAY = 0.01
NUM_EPOCHS = 4
WARMUP_RATIO = 0.10
EARLY_STOPPING_PATIENCE = 2
USE_FP16 = True             # NVIDIA T4 natively supports fp16 (do NOT set bf16=True on T4)

# Loss Function Option: "cross_entropy", "weighted_ce", or "focal_loss"
LOSS_FUNCTION_TYPE = "cross_entropy"
FOCAL_GAMMA = 2.0

# PubMed 200k RCT Subsetting (To stay comfortably within Kaggle 12h session limit)
# Set to an integer (e.g. 50000) or None for full 2.2M sentences
PUBMED_200K_TRAIN_SAMPLES = 50000

# Output paths
WORKING_DIR = "/kaggle/working" if os.path.exists("/kaggle/working") else "./output"
MODELS_OUTPUT_DIR = os.path.join(WORKING_DIR, "models")
os.makedirs(MODELS_OUTPUT_DIR, exist_ok=True)

Using processed data directory: /kaggle/input/datasets/siamabir/rhetorical-roles-processed-data/processed_data


### Step 3: Kaggle Input Directory Listing
Verify the exact contents of `/kaggle/input/` and the processed data folder.

In [3]:
print("=" * 80)
print("INSPECTING INPUT DATA FILES")
print("=" * 80)

if DATA_DIR and os.path.exists(DATA_DIR):
    for root, dirs, files in os.walk(DATA_DIR):
        for fname in sorted(files):
            fpath = os.path.join(root, fname)
            size_mb = os.path.getsize(fpath) / (1024 * 1024)
            rel_path = os.path.relpath(fpath, DATA_DIR)
            print(f"  [FILE] {rel_path:42s} | Size: {size_mb:6.2f} MB")
else:
    print(f"WARNING: Data directory not found: {DATA_DIR}")
    print("Please run 01_data_preprocessing.ipynb first or attach the dataset.")

INSPECTING INPUT DATA FILES
  [FILE] combined_sentences.parquet                 | Size:  68.71 MB
  [FILE] dataset_metadata.json                      | Size:   0.00 MB
  [FILE] llm_instruction_5k.jsonl                   | Size:  77.66 MB
  [FILE] llm_instruction_pubmed20k.jsonl            | Size: 277.30 MB
  [FILE] manifest.json                              | Size:   0.00 MB
  [FILE] medical_abstracts_5k_sentences.csv         | Size:  32.20 MB
  [FILE] medical_abstracts_5k_sentences.parquet     | Size:  14.20 MB
  [FILE] pubmed_200k_sentences.parquet              | Size: 505.58 MB
  [FILE] pubmed_20k_sentences.csv                   | Size: 113.61 MB
  [FILE] pubmed_20k_sentences.parquet               | Size:  54.55 MB


### Step 4: Discourse Dataset Class & Feature Formatting
Here we formulate the input representation per the thesis plan:
- **Input Text:** `[CLS] Target Sentence [SEP] Context: Preceding: <prev> | Succeeding: <next> [SEP]`
- **Positional Features:** Continuous vector $\mathbf{p} = [\text{normalized\_pos}, \frac{\text{line\_number}}{\text{max\_lines}}, \frac{\text{total\_lines}}{\text{max\_lines}}] \in \mathbb{R}^3$
- **Target Label:** Integer ID in $\{0, 1, 2, 3, 4\}$ corresponding to `BACKGROUND`, `OBJECTIVE`, `METHODS`, `RESULTS`, `CONCLUSION`.

In [4]:
class BiomedicalDiscourseDataset(Dataset):
    """
    PyTorch Dataset for Sequential Rhetorical Role Classification.
    Constructs contextual sentence representations and extracts continuous positional features.
    """
    def __init__(self, df: pd.DataFrame, tokenizer, max_length: int = 256):
        self.tokenizer = tokenizer
        self.max_length = max_length
        self.labels = [LABEL2ID[lbl] for lbl in df['label'].values]
        
        # Prepare textual sequences with context
        # Format: Target Sentence [SEP] Context: Preceding: <prev> | Succeeding: <next>
        self.target_sentences = df['sentence_text'].astype(str).tolist()
        self.prev_sentences = df['prev_sentence'].astype(str).tolist()
        self.next_sentences = df['next_sentence'].astype(str).tolist()
        
        # Extract continuous positional features
        norm_pos = df['normalized_pos'].astype(float).values
        line_no = (df['line_number'].astype(float) / 50.0).values  # Normalization scaling
        total_l = (df['total_lines'].astype(float) / 50.0).values
        
        self.pos_features = np.stack([norm_pos, line_no, total_l], axis=1).astype(np.float32)
        
    def __len__(self):
        return len(self.labels)
        
    def __getitem__(self, idx: int):
        target_s = self.target_sentences[idx]
        prev_s = self.prev_sentences[idx]
        next_s = self.next_sentences[idx]
        
        # Dual-sentence context window encoding
        context_str = f"Context: Preceding: {prev_s} | Succeeding: {next_s}"
        
        encoding = self.tokenizer(
            target_s,
            context_str,
            truncation=True,
            max_length=self.max_length,
            padding="max_length",
            return_tensors="pt"
        )
        
        return {
            "input_ids": encoding["input_ids"].squeeze(0),
            "attention_mask": encoding["attention_mask"].squeeze(0),
            "token_type_ids": encoding.get("token_type_ids", torch.zeros_like(encoding["attention_mask"])).squeeze(0),
            "pos_features": torch.tensor(self.pos_features[idx], dtype=torch.float32),
            "label": torch.tensor(self.labels[idx], dtype=torch.long)
        }

print("BiomedicalDiscourseDataset class successfully initialized.")

BiomedicalDiscourseDataset class successfully initialized.


### Step 5: Model Architecture: Dual-Stream Context & Positional Fusion Head
We implement `BiomedicalDiscourseClassifier(nn.Module)`:
1. **Biomedical Transformer Encoder:** Encodes `[CLS] Target [SEP] Context [SEP]` into $d_{\text{hidden}}$ (768-d text representation).
2. **Positional Feature Projection:** Projects the 3-dimensional positional vector $[\text{norm\_pos}, \text{line\_no}/50, \text{total\_lines}/50]$ through a Linear layer $\to$ GELU $\to$ LayerNorm $\to 64$-d positional embedding.
3. **Multimodal Fusion & Classification:** Concatenates $[\mathbf{h}_{\text{text}}; \mathbf{h}_{\text{pos}}] \in \mathbb{R}^{768 + 64 = 832}$, applies Dropout(0.2), and passes through a Linear projection to compute the 5 class logits.

In [5]:
class FocalLoss(nn.Module):
    """Focal Loss for addressing class imbalance by down-weighting easy examples."""
    def __init__(self, alpha: Optional[torch.Tensor] = None, gamma: float = 2.0):
        super(FocalLoss, self).__init__()
        self.alpha = alpha
        self.gamma = gamma

    def forward(self, inputs: torch.Tensor, targets: torch.Tensor) -> torch.Tensor:
        ce_loss = F.cross_entropy(inputs, targets, reduction='none', weight=self.alpha)
        pt = torch.exp(-ce_loss)
        focal_loss = ((1.0 - pt) ** self.gamma) * ce_loss
        return focal_loss.mean()

class BiomedicalDiscourseClassifier(nn.Module):
    """
    Dual-stream biomedical encoder integrating contextualized text embeddings
    with continuous rhetorical position projections.
    """
    def __init__(self, model_id: str, num_classes: int = 5, pos_dim: int = 3, pos_hidden: int = 64, dropout_rate: float = 0.2):
        super(BiomedicalDiscourseClassifier, self).__init__()
        self.encoder = AutoModel.from_pretrained(model_id)
        hidden_size = self.encoder.config.hidden_size
        
        # Positional feature projection network
        self.pos_proj = nn.Sequential(
            nn.Linear(pos_dim, pos_hidden),
            nn.GELU(),
            nn.LayerNorm(pos_hidden)
        )
        
        # Classification head combining text representation + positional representation
        self.dropout = nn.Dropout(dropout_rate)
        self.classifier = nn.Linear(hidden_size + pos_hidden, num_classes)
        
    def forward(self, input_ids, attention_mask, pos_features, token_type_ids=None):
        # 1. Text stream
        outputs = self.encoder(input_ids=input_ids, attention_mask=attention_mask)
        # Extract pooled [CLS] token embedding
        cls_rep = outputs.last_hidden_state[:, 0, :]
        
        # 2. Position stream
        pos_rep = self.pos_proj(pos_features)
        
        # 3. Multimodal fusion
        fused = torch.cat([cls_rep, pos_rep], dim=1)
        fused = self.dropout(fused)
        logits = self.classifier(fused)
        return logits

print("BiomedicalDiscourseClassifier module defined.")

BiomedicalDiscourseClassifier module defined.


### Step 6: Phase 4 Comprehensive Evaluation Framework
Implements all evaluation metrics specified in the thesis plan:
1. **Overall Accuracy, Macro-F1, Micro-F1**
2. **Per-class Precision, Recall, F1** for all 5 roles
3. **5x5 Confusion Matrix** saved as CSV and high-resolution PNG heatmap
4. **Sequential Transition Consistency:** Defines and computes the empirical transition probability matrix $P(y_t \mid y_{t-1})$ and measures the fraction of transitions that follow monotonic forward discourse progression ($\text{rank}(y_t) \ge \text{rank}(y_{t-1})$).

In [6]:
import os
import sys
import json
import torch
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import seaborn as sns
from typing import Dict, List, Tuple, Any, Optional
from sklearn.metrics import accuracy_score, f1_score, precision_recall_fscore_support, confusion_matrix

def compute_sequential_transition_consistency(df_test: pd.DataFrame, y_pred: np.ndarray) -> Tuple[float, np.ndarray, np.ndarray]:
    """
    Quantifies sequential transition consistency across consecutive sentences within abstracts.
    A transition (y_{t-1} -> y_t) is legal (monotonic) if rank(y_t) >= rank(y_{t-1}).
    """
    df_eval = df_test.copy()
    df_eval['pred_label_id'] = y_pred
    df_eval['true_label_id'] = [LABEL2ID[lbl] for lbl in df_eval['label'].values]
    
    total_consecutive_pairs = 0
    legal_transitions_pred = 0
    
    trans_matrix_true = np.zeros((NUM_CLASSES, NUM_CLASSES), dtype=np.int64)
    trans_matrix_pred = np.zeros((NUM_CLASSES, NUM_CLASSES), dtype=np.int64)
    
    for pid, group in df_eval.groupby('paper_id'):
        group_sorted = group.sort_values('line_number')
        true_ids = group_sorted['true_label_id'].values
        pred_ids = group_sorted['pred_label_id'].values
        
        for i in range(1, len(true_ids)):
            t_prev, t_curr = true_ids[i-1], true_ids[i]
            p_prev, p_curr = pred_ids[i-1], pred_ids[i]
            
            trans_matrix_true[t_prev, t_curr] += 1
            trans_matrix_pred[p_prev, p_curr] += 1
            
            total_consecutive_pairs += 1
            if p_curr >= p_prev:
                legal_transitions_pred += 1
                
    consistency_rate = (legal_transitions_pred / total_consecutive_pairs * 100.0) if total_consecutive_pairs > 0 else 0.0
    return consistency_rate, trans_matrix_true, trans_matrix_pred

def evaluate_model_comprehensive(model, dataloader, df_test, device, output_dir, run_name):
    """Runs comprehensive evaluation on test set, logs metrics, generates confusion matrix and transition analysis."""
    model.eval()
    all_preds = []
    all_targets = []
    
    with torch.no_grad():
        for batch in dataloader:
            input_ids = batch["input_ids"].to(device)
            attention_mask = batch["attention_mask"].to(device)
            pos_features = batch["pos_features"].to(device)
            labels = batch["label"].to(device)
            
            with torch.cuda.amp.autocast(enabled=USE_FP16):
                logits = model(input_ids=input_ids, attention_mask=attention_mask, pos_features=pos_features)
                preds = torch.argmax(logits, dim=1)
                
            all_preds.extend(preds.cpu().numpy())
            all_targets.extend(labels.cpu().numpy())
            
    all_preds = np.array(all_preds)
    all_targets = np.array(all_targets)
    
    acc = accuracy_score(all_targets, all_preds)
    macro_f1 = f1_score(all_targets, all_preds, average="macro")
    micro_f1 = f1_score(all_targets, all_preds, average="micro")
    
    p_per_class, r_per_class, f1_per_class, _ = precision_recall_fscore_support(all_targets, all_preds, labels=list(range(NUM_CLASSES)), zero_division=0)
    
    # Sequential transition consistency
    trans_consistency, tm_true, tm_pred = compute_sequential_transition_consistency(df_test, all_preds)
    
    # 5x5 Confusion Matrix
    cm = confusion_matrix(all_targets, all_preds, labels=list(range(NUM_CLASSES)))
    
    # Save Confusion Matrix Heatmap
    os.makedirs(output_dir, exist_ok=True)
    cm_img_path = os.path.join(output_dir, f"{run_name}_confusion_matrix.png")
    plt.figure(figsize=(7, 6))
    sns.heatmap(cm, annot=True, fmt="d", cmap="Blues", xticklabels=TARGET_CLASSES, yticklabels=TARGET_CLASSES)
    plt.title(f"Confusion Matrix: {run_name}\nMacro F1: {macro_f1*100:.2f}% | Acc: {acc*100:.2f}%")
    plt.xlabel("Predicted Role")
    plt.ylabel("True Rhetorical Role")
    plt.tight_layout()
    plt.savefig(cm_img_path, dpi=300)
    plt.close()
    
    # Save Confusion Matrix CSV
    cm_df = pd.DataFrame(cm, index=[f"True_{c}" for c in TARGET_CLASSES], columns=[f"Pred_{c}" for c in TARGET_CLASSES])
    cm_csv_path = os.path.join(output_dir, f"{run_name}_confusion_matrix.csv")
    cm_df.to_csv(cm_csv_path)
    
    # Package metrics
    metrics = {
        "run_name": run_name,
        "accuracy": float(acc),
        "macro_f1": float(macro_f1),
        "micro_f1": float(micro_f1),
        "sequential_transition_consistency_pct": float(trans_consistency),
        "per_class": {
            TARGET_CLASSES[i]: {
                "precision": float(p_per_class[i]),
                "recall": float(r_per_class[i]),
                "f1": float(f1_per_class[i])
            } for i in range(NUM_CLASSES)
        }
    }
    
    metrics_json_path = os.path.join(output_dir, f"{run_name}_metrics.json")
    with open(metrics_json_path, "w", encoding="utf-8") as f:
        json.dump(metrics, f, indent=2)
        
    print(f"\n{'='*70}")
    print(f"EVALUATION RESULTS: {run_name}")
    print(f"{'='*70}")
    print(f"  Accuracy:      {acc*100:.2f}%")
    print(f"  Macro-F1:      {macro_f1*100:.2f}%")
    print(f"  Micro-F1:      {micro_f1*100:.2f}%")
    print(f"  Transition Consistency: {trans_consistency:.2f}%")
    print("  Per-Class F1 Scores:")
    for i, c in enumerate(TARGET_CLASSES):
        print(f"    {c:12s}: Precision={p_per_class[i]*100:5.2f}% | Recall={r_per_class[i]*100:5.2f}% | F1={f1_per_class[i]*100:5.2f}%")
    print(f"  Artifacts saved to: {output_dir}")
    
    return metrics

### Step 7: Multi-GPU Training Loop with Best Checkpoint & Resumption
We define `train_model()`:
- Supports multi-GPU via `nn.DataParallel` on Kaggle T4 x2
- Native fp16 mixed-precision using `torch.cuda.amp.autocast()` and `GradScaler()`
- Tracks validation Macro-F1 every epoch and saves the best model state dictionary
- Implements early stopping patience to avoid overfitting
- Reports training throughput and loss progression

In [7]:
import os
import sys
import json
import random
import torch
import torch.nn as nn
import numpy as np
import pandas as pd
from typing import Dict, List, Tuple, Any, Optional
from transformers import AutoTokenizer, get_cosine_schedule_with_warmup
from sklearn.metrics import accuracy_score, f1_score

def train_model(model_info: Dict[str, str], df_train: pd.DataFrame, df_val: pd.DataFrame, df_test: pd.DataFrame, dataset_name: str) -> Dict[str, Any]:
    """Trains a biomedical discourse classifier, tracking validation Macro-F1 and saving best weights."""
    model_id = model_info["model_id"]
    model_short_name = model_info["name"]
    run_name = f"{dataset_name}_{model_short_name}"
    run_output_dir = os.path.join(MODELS_OUTPUT_DIR, dataset_name, model_short_name)
    os.makedirs(run_output_dir, exist_ok=True)
    
    print(f"\n{'#'*80}")
    print(f"STARTING EXPERIMENT: {run_name}")
    print(f"  Model ID:     {model_id}")
    print(f"  Dataset:      {dataset_name} (Train={len(df_train):,}, Val={len(df_val):,}, Test={len(df_test):,})")
    print(f"{'#'*80}")
    
    # 1. Initialize Tokenizer & Datasets
    tokenizer = AutoTokenizer.from_pretrained(model_id)
    ds_train = BiomedicalDiscourseDataset(df_train, tokenizer, max_length=MAX_SEQ_LENGTH)
    ds_val = BiomedicalDiscourseDataset(df_val, tokenizer, max_length=MAX_SEQ_LENGTH)
    ds_test = BiomedicalDiscourseDataset(df_test, tokenizer, max_length=MAX_SEQ_LENGTH)
    
    num_gpus = torch.cuda.device_count() if torch.cuda.is_available() else 1
    eff_batch_size = BATCH_SIZE_PER_GPU * max(1, num_gpus)
    
    train_loader = DataLoader(ds_train, batch_size=eff_batch_size, shuffle=True, num_workers=2, pin_memory=True)
    val_loader = DataLoader(ds_val, batch_size=eff_batch_size, shuffle=False, num_workers=2, pin_memory=True)
    test_loader = DataLoader(ds_test, batch_size=eff_batch_size, shuffle=False, num_workers=2, pin_memory=True)
    
    # 2. Instantiate Model & Hardware Placement
    device = torch.device("cuda:0" if torch.cuda.is_available() else "cpu")
    model = BiomedicalDiscourseClassifier(model_id=model_id, num_classes=NUM_CLASSES)
    model.to(device)
    
    if num_gpus > 1:
        print(f"Enabling PyTorch DataParallel across {num_gpus} GPUs (cuda:0, cuda:1).")
        model_parallel = nn.DataParallel(model)
    else:
        model_parallel = model
        
    # 3. Loss Function Configuration
    if LOSS_FUNCTION_TYPE == "focal_loss":
        criterion = FocalLoss(gamma=FOCAL_GAMMA)
        print("Using Focal Loss (gamma=2.0).")
    elif LOSS_FUNCTION_TYPE == "weighted_ce":
        class_counts = df_train['label'].map(LABEL2ID).value_counts().sort_index().values
        weights = torch.tensor(len(df_train) / (NUM_CLASSES * class_counts), dtype=torch.float32).to(device)
        criterion = nn.CrossEntropyLoss(weight=weights)
        print(f"Using Weighted Cross-Entropy Loss with class weights: {weights.cpu().numpy().round(3)}")
    else:
        criterion = nn.CrossEntropyLoss()
        print("Using Standard Cross-Entropy Loss.")
        
    # 4. Optimizer & Cosine LR Scheduler
    no_decay = ['bias', 'LayerNorm.weight']
    optimizer_grouped_parameters = [
        {'params': [p for n, p in model.named_parameters() if not any(nd in n for nd in no_decay)], 'weight_decay': WEIGHT_DECAY},
        {'params': [p for n, p in model.named_parameters() if any(nd in n for nd in no_decay)], 'weight_decay': 0.0}
    ]
    optimizer = torch.optim.AdamW(optimizer_grouped_parameters, lr=LEARNING_RATE)
    
    total_training_steps = len(train_loader) * NUM_EPOCHS
    warmup_steps = int(total_training_steps * WARMUP_RATIO)
    scheduler = get_cosine_schedule_with_warmup(optimizer, num_warmup_steps=warmup_steps, num_training_steps=total_training_steps)
    scaler = torch.cuda.amp.GradScaler(enabled=USE_FP16)
    
    # 5. Training Loop
    best_val_macro_f1 = -1.0
    patience_counter = 0
    best_model_path = os.path.join(run_output_dir, "best_model.pt")
    
    for epoch in range(1, NUM_EPOCHS + 1):
        model.train()
        total_loss = 0.0
        print(f"\n--- Epoch {epoch}/{NUM_EPOCHS} ---")
        
        for step, batch in enumerate(train_loader, 1):
            input_ids = batch["input_ids"].to(device)
            attention_mask = batch["attention_mask"].to(device)
            pos_features = batch["pos_features"].to(device)
            labels = batch["label"].to(device)
            
            optimizer.zero_grad()
            with torch.cuda.amp.autocast(enabled=USE_FP16):
                logits = model_parallel(input_ids=input_ids, attention_mask=attention_mask, pos_features=pos_features)
                loss = criterion(logits, labels)
                
            scaler.scale(loss).backward()
            scaler.unscale_(optimizer)
            torch.nn.utils.clip_grad_norm_(model.parameters(), 1.0)
            scaler.step(optimizer)
            scaler.update()
            scheduler.step()
            
            total_loss += loss.item()
            if step % 100 == 0 or step == len(train_loader):
                print(f"  Step {step:4d}/{len(train_loader):4d} | Avg Loss: {total_loss/step:6.4f} | LR: {scheduler.get_last_lr()[0]:.2e}")
                
        # Epoch Validation
        model.eval()
        val_preds, val_targets = [], []
        with torch.no_grad():
            for batch in val_loader:
                input_ids = batch["input_ids"].to(device)
                attention_mask = batch["attention_mask"].to(device)
                pos_features = batch["pos_features"].to(device)
                labels = batch["label"].to(device)
                
                with torch.cuda.amp.autocast(enabled=USE_FP16):
                    logits = model_parallel(input_ids=input_ids, attention_mask=attention_mask, pos_features=pos_features)
                    preds = torch.argmax(logits, dim=1)
                val_preds.extend(preds.cpu().numpy())
                val_targets.extend(labels.cpu().numpy())
                
        val_macro_f1 = f1_score(val_targets, val_preds, average="macro")
        val_acc = accuracy_score(val_targets, val_preds)
        print(f"  --> Epoch {epoch} Validation: Macro-F1 = {val_macro_f1*100:.2f}% | Accuracy = {val_acc*100:.2f}%")
        
        # Checkpoint Best Model
        if val_macro_f1 > best_val_macro_f1:
            best_val_macro_f1 = val_macro_f1
            patience_counter = 0
            # Save un-wrapped state dict
            raw_model = model.module if hasattr(model, "module") else model
            torch.save(raw_model.state_dict(), best_model_path)
            tokenizer.save_pretrained(run_output_dir)
            print(f"  [SAVED] New best model saved to {best_model_path} (Val Macro-F1: {val_macro_f1*100:.2f}%)")
        else:
            patience_counter += 1
            print(f"  Validation score did not improve. Early stopping counter: {patience_counter}/{EARLY_STOPPING_PATIENCE}")
            if patience_counter >= EARLY_STOPPING_PATIENCE:
                print(f"  Early stopping triggered at Epoch {epoch}.")
                break
                
    # 6. Final Evaluation on Unseen Test Split using Best Weights
    print(f"\nLoading best weights for final test evaluation: {best_model_path}")
    raw_model = model.module if hasattr(model, "module") else model
    raw_model.load_state_dict(torch.load(best_model_path, map_location=device))
    test_metrics = evaluate_model_comprehensive(raw_model, test_loader, df_test, device, run_output_dir, run_name)
    
    # Save Model Card
    card = {
        "model_name": model_short_name,
        "base_model_id": model_id,
        "dataset_name": dataset_name,
        "train_samples": len(df_train),
        "test_metrics": test_metrics,
        "hyperparameters": {
            "max_seq_length": MAX_SEQ_LENGTH,
            "effective_batch_size": eff_batch_size,
            "learning_rate": LEARNING_RATE,
            "epochs_trained": epoch,
            "loss_function": LOSS_FUNCTION_TYPE,
            "fp16": USE_FP16
        }
    }
    with open(os.path.join(run_output_dir, "model_card.json"), "w", encoding="utf-8") as f:
        json.dump(card, f, indent=2)
        
    return test_metrics

### Step 8: Load Preprocessed Data & Hardware Reality Benchmark
Load `medical_abstracts_5k`, `pubmed_20k_rct`, and check `pubmed_200k_rct`.
Run a 50-step benchmark pass to measure real steps/second and print estimated total time.

In [8]:
import random

print("=" * 80)
print("LOADING DATASETS FROM PROCESSED ARTIFACTS")
print("=" * 80)

# 1. Dataset 1 (5k Abstracts)
p_5k = find_processed_file("medical_abstracts_5k_sentences.parquet", DATA_DIR)
df_5k = pd.read_parquet(p_5k)
df_5k_train = df_5k[df_5k['split'] == 'train'].reset_index(drop=True)
df_5k_val = df_5k[df_5k['split'] == 'val'].reset_index(drop=True)
df_5k_test = df_5k[df_5k['split'] == 'test'].reset_index(drop=True)
print(f"Loaded medical_abstracts_5k: Train={len(df_5k_train):,}, Val={len(df_5k_val):,}, Test={len(df_5k_test):,}")

# 2. Dataset 2 (PubMed 20k)
p_20k = find_processed_file("pubmed_20k_sentences.parquet", DATA_DIR)
df_20k = pd.read_parquet(p_20k)
df_20k_train = df_20k[df_20k['split'] == 'train'].reset_index(drop=True)
df_20k_val = df_20k[df_20k['split'] == 'val'].reset_index(drop=True)
df_20k_test = df_20k[df_20k['split'] == 'test'].reset_index(drop=True)
print(f"Loaded pubmed_20k_rct: Train={len(df_20k_train):,}, Val={len(df_20k_val):,}, Test={len(df_20k_test):,}")

# 3. Dataset 3 (PubMed 200k with explicit configuration limit)
try:
    p_200k = find_processed_file("pubmed_200k_sentences.parquet", DATA_DIR)
except FileNotFoundError:
    p_200k = None

if p_200k and os.path.exists(p_200k):
    df_200k = pd.read_parquet(p_200k)
    df_200k_train_full = df_200k[df_200k['split'] == 'train'].reset_index(drop=True)
    df_200k_val = df_200k[df_200k['split'] == 'val'].reset_index(drop=True)
    df_200k_test = df_200k[df_200k['split'] == 'test'].reset_index(drop=True)
    
    if PUBMED_200K_TRAIN_SAMPLES is not None and PUBMED_200K_TRAIN_SAMPLES < len(df_200k_train_full):
        # Deterministic paper-grouped subsampling
        pids_pool = df_200k_train_full['paper_id'].unique()
        random.seed(RANDOM_SEED)
        sampled_pids = set(random.sample(list(pids_pool), int(PUBMED_200K_TRAIN_SAMPLES / 11.6)))
        df_200k_train = df_200k_train_full[df_200k_train_full['paper_id'].isin(sampled_pids)].reset_index(drop=True)
        print(f"PubMed 200k Subsampled to {len(df_200k_train):,} training sentences (across {len(sampled_pids):,} papers) per PUBMED_200K_TRAIN_SAMPLES configuration.")
    else:
        df_200k_train = df_200k_train_full
        print(f"PubMed 200k using full train set: {len(df_200k_train):,} sentences.")
else:
    df_200k_train, df_200k_val, df_200k_test = None, None, None
    print("PubMed 200k parquet not found; skipping 200k training.")

LOADING DATASETS FROM PROCESSED ARTIFACTS
Loaded medical_abstracts_5k: Train=45,302, Val=9,730, Test=9,845
Loaded pubmed_20k_rct: Train=180,040, Val=30,212, Test=30,135
PubMed 200k Subsampled to 50,176 training sentences (across 4,310 papers) per PUBMED_200K_TRAIN_SAMPLES configuration.


### Step 9: Training & Evaluation on Dataset 1 (`medical_abstracts_5k`)
Execute training for both biomedical encoders (`PubMedBERT-base` and `BioLinkBERT-base`) on the 5k Medical Abstracts dataset.

In [9]:
all_experiment_results = []

for model_cfg in MODELS_TO_TRAIN:
    metrics = train_model(
        model_info=model_cfg,
        df_train=df_5k_train,
        df_val=df_5k_val,
        df_test=df_5k_test,
        dataset_name="medical_abstracts_5k"
    )
    all_experiment_results.append(metrics)
    torch.cuda.empty_cache()


################################################################################
STARTING EXPERIMENT: medical_abstracts_5k_PubMedBERT-base
  Model ID:     microsoft/BiomedNLP-PubMedBERT-base-uncased-abstract
  Dataset:      medical_abstracts_5k (Train=45,302, Val=9,730, Test=9,845)
################################################################################


config.json:   0%|          | 0.00/385 [00:00<?, ?B/s]

tokenizer_config.json:   0%|          | 0.00/28.0 [00:00<?, ?B/s]

vocab.txt:   0%|          | 0.00/225k [00:00<?, ?B/s]

pytorch_model.bin: reconstructing file:   0%|          |  0.00B /  440MB            

...

  Step  100/ 708 | Avg Loss: 0.1910 | LR: 4.02e-06
  Step  200/ 708 | Avg Loss: 0.1861 | LR: 2.85e-06
  Step  300/ 708 | Avg Loss: 0.1843 | LR: 1.86e-06
  Step  400/ 708 | Avg Loss: 0.1833 | LR: 1.07e-06
  Step  500/ 708 | Avg Loss: 0.1843 | LR: 4.90e-07
  Step  600/ 708 | Avg Loss: 0.1838 | LR: 1.33e-07
  Step  700/ 708 | Avg Loss: 0.1827 | LR: 7.29e-10
  Step  708/ 708 | Avg Loss: 0.1825 | LR: 0.00e+00


/tmp/ipykernel_22/2114198636.py:119: FutureWarning: `torch.cuda.amp.autocast(args...)` is deprecated. Please use `torch.amp.autocast('cuda', args...)` instead.
  with torch.cuda.amp.autocast(enabled=USE_FP16):


  --> Epoch 4 Validation: Macro-F1 = 83.15% | Accuracy = 83.70%
  Validation score did not improve. Early stopping counter: 1/2

Loading best weights for final test evaluation: /kaggle/working/models/medical_abstracts_5k/BioLinkBERT-base/best_model.pt


/tmp/ipykernel_22/144664855.py:59: FutureWarning: `torch.cuda.amp.autocast(args...)` is deprecated. Please use `torch.amp.autocast('cuda', args...)` instead.
  with torch.cuda.amp.autocast(enabled=USE_FP16):



EVALUATION RESULTS: medical_abstracts_5k_BioLinkBERT-base
  Accuracy:      84.03%
  Macro-F1:      83.43%
  Micro-F1:      84.03%
  Transition Consistency: 95.74%
  Per-Class F1 Scores:
    BACKGROUND  : Precision=85.32% | Recall=85.91% | F1=85.61%
    OBJECTIVE   : Precision=81.53% | Recall=73.63% | F1=77.38%
    METHODS     : Precision=82.16% | Recall=80.73% | F1=81.44%
    RESULTS     : Precision=83.80% | Recall=88.32% | F1=86.00%
    CONCLUSION  : Precision=87.76% | Recall=85.71% | F1=86.72%
  Artifacts saved to: /kaggle/working/models/medical_abstracts_5k/BioLinkBERT-base


### Step 10: Training & Evaluation on Dataset 2 (`PubMed_20k_RCT`)
Execute training on the standard 20k RCT benchmark dataset.

In [10]:
for model_cfg in MODELS_TO_TRAIN:
    metrics = train_model(
        model_info=model_cfg,
        df_train=df_20k_train,
        df_val=df_20k_val,
        df_test=df_20k_test,
        dataset_name="pubmed_20k_rct"
    )
    all_experiment_results.append(metrics)
    torch.cuda.empty_cache()


################################################################################
STARTING EXPERIMENT: pubmed_20k_rct_PubMedBERT-base
  Model ID:     microsoft/BiomedNLP-PubMedBERT-base-uncased-abstract
  Dataset:      pubmed_20k_rct (Train=180,040, Val=30,212, Test=30,135)
################################################################################


Loading weights:   0%|          | 0/199 [00:00<?, ?it/s]

[transformers] BertModel LOAD REPORT from: microsoft/BiomedNLP-PubMedBERT-base-uncased-abstract
Key                                        | Status     |  | 
-------------------------------------------+------------+--+-
cls.seq_relationship.bias                  | UNEXPECTED |  | 
cls.predictions.transform.LayerNorm.bias   | UNEXPECTED |  | 
cls.predictions.decoder.bias               | UNEXPECTED |  | 
cls.predictions.transform.LayerNorm.weight | UNEXPECTED |  | 
cls.predictions.transform.dense.weight     | UNEXPECTED |  | 
cls.predictions.transform.dense.bias       | UNEXPECTED |  | 
cls.predictions.bias                       | UNEXPECTED |  | 
cls.predictions.decoder.weight             | UNEXPECTED |  | 
cls.seq_relationship.weight                | UNEXPECTED |  | 

Notes:
- UNEXPECTED:	can be ignored when loading from different task/architecture; not ok if you expect identical arch.


Enabling PyTorch DataParallel across 2 GPUs (cuda:0, cuda:1).
Using Standard Cross-Entropy Loss.

--- Epoch 1/4 ---


/tmp/ipykernel_22/2114198636.py:75: FutureWarning: `torch.cuda.amp.GradScaler(args...)` is deprecated. Please use `torch.amp.GradScaler('cuda', args...)` instead.
  scaler = torch.cuda.amp.GradScaler(enabled=USE_FP16)
/tmp/ipykernel_22/2114198636.py:94: FutureWarning: `torch.cuda.amp.autocast(args...)` is deprecated. Please use `torch.amp.autocast('cuda', args...)` instead.
  with torch.cuda.amp.autocast(enabled=USE_FP16):
/tmp/ipykernel_22/2114198636.py:103: UserWarning: Detected call of `lr_scheduler.step()` before `optimizer.step()`. In PyTorch 1.1.0 and later, you should call them in the opposite order: `optimizer.step()` before `lr_scheduler.step()`.  Failure to do this will result in PyTorch skipping the first value of the learning rate schedule. See more details at https://pytorch.org/docs/stable/optim.html#how-to-adjust-learning-rate
  scheduler.step()


...

  Step  100/2814 | Avg Loss: 0.0892 | LR: 5.01e-06
  Step  200/2814 | Avg Loss: 0.0861 | LR: 4.66e-06
  Step  300/2814 | Avg Loss: 0.0861 | LR: 4.33e-06
  Step  400/2814 | Avg Loss: 0.0888 | LR: 4.01e-06
  Step  500/2814 | Avg Loss: 0.0879 | LR: 3.70e-06
  Step  600/2814 | Avg Loss: 0.0865 | LR: 3.40e-06
  Step  700/2814 | Avg Loss: 0.0861 | LR: 3.11e-06
  Step  800/2814 | Avg Loss: 0.0857 | LR: 2.83e-06
  Step  900/2814 | Avg Loss: 0.0845 | LR: 2.57e-06
  Step 1000/2814 | Avg Loss: 0.0845 | LR: 2.31e-06
  Step 1100/2814 | Avg Loss: 0.0840 | LR: 2.07e-06
  Step 1200/2814 | Avg Loss: 0.0834 | LR: 1.84e-06
  Step 1300/2814 | Avg Loss: 0.0841 | LR: 1.62e-06
  Step 1400/2814 | Avg Loss: 0.0834 | LR: 1.42e-06
  Step 1500/2814 | Avg Loss: 0.0834 | LR: 1.23e-06
  Step 1600/2814 | Avg Loss: 0.0836 | LR: 1.05e-06
  Step 1700/2814 | Avg Loss: 0.0833 | LR: 8.86e-07
  Step 1800/2814 | Avg Loss: 0.0829 | LR: 7.35e-07
  Step 1900/2814 | Avg Loss: 0.0824 | LR: 5.98e-07
  Step 2000/2814 | Avg Loss: 0.

/tmp/ipykernel_22/2114198636.py:119: FutureWarning: `torch.cuda.amp.autocast(args...)` is deprecated. Please use `torch.amp.autocast('cuda', args...)` instead.
  with torch.cuda.amp.autocast(enabled=USE_FP16):


  --> Epoch 4 Validation: Macro-F1 = 88.02% | Accuracy = 92.74%
  Validation score did not improve. Early stopping counter: 1/2

Loading best weights for final test evaluation: /kaggle/working/models/pubmed_20k_rct/BioLinkBERT-base/best_model.pt


/tmp/ipykernel_22/144664855.py:59: FutureWarning: `torch.cuda.amp.autocast(args...)` is deprecated. Please use `torch.amp.autocast('cuda', args...)` instead.
  with torch.cuda.amp.autocast(enabled=USE_FP16):



EVALUATION RESULTS: pubmed_20k_rct_BioLinkBERT-base
  Accuracy:      92.52%
  Macro-F1:      88.06%
  Micro-F1:      92.52%
  Transition Consistency: 96.85%
  Per-Class F1 Scores:
    BACKGROUND  : Precision=77.88% | Recall=85.67% | F1=81.59%
    OBJECTIVE   : Precision=76.41% | Recall=64.85% | F1=70.16%
    METHODS     : Precision=95.62% | Recall=97.45% | F1=96.53%
    RESULTS     : Precision=97.11% | Recall=95.02% | F1=96.05%
    CONCLUSION  : Precision=95.87% | Recall=96.08% | F1=95.98%
  Artifacts saved to: /kaggle/working/models/pubmed_20k_rct/BioLinkBERT-base


### Step 11: Training & Evaluation on Dataset 3 (`PubMed_200k_RCT`)
If configured and available, train on the PubMed 200k RCT dataset.

In [11]:
if df_200k_train is not None and len(df_200k_train) > 0:
    for model_cfg in MODELS_TO_TRAIN[:1]:  # Train primary model (PubMedBERT) on 200k
        metrics = train_model(
            model_info=model_cfg,
            df_train=df_200k_train,
            df_val=df_200k_val,
            df_test=df_200k_test,
            dataset_name="pubmed_200k_rct"
        )
        all_experiment_results.append(metrics)
        torch.cuda.empty_cache()
else:
    print("PubMed 200k dataset was skipped or not available.")


################################################################################
STARTING EXPERIMENT: pubmed_200k_rct_PubMedBERT-base
  Model ID:     microsoft/BiomedNLP-PubMedBERT-base-uncased-abstract
  Dataset:      pubmed_200k_rct (Train=50,176, Val=28,932, Test=29,493)
################################################################################


Loading weights:   0%|          | 0/199 [00:00<?, ?it/s]

[transformers] BertModel LOAD REPORT from: microsoft/BiomedNLP-PubMedBERT-base-uncased-abstract
Key                                        | Status     |  | 
-------------------------------------------+------------+--+-
cls.seq_relationship.bias                  | UNEXPECTED |  | 
cls.predictions.transform.LayerNorm.bias   | UNEXPECTED |  | 
cls.predictions.decoder.bias               | UNEXPECTED |  | 
cls.predictions.transform.LayerNorm.weight | UNEXPECTED |  | 
cls.predictions.transform.dense.weight     | UNEXPECTED |  | 
cls.predictions.transform.dense.bias       | UNEXPECTED |  | 
cls.predictions.bias                       | UNEXPECTED |  | 
cls.predictions.decoder.weight             | UNEXPECTED |  | 
cls.seq_relationship.weight                | UNEXPECTED |  | 

Notes:
- UNEXPECTED:	can be ignored when loading from different task/architecture; not ok if you expect identical arch.


Enabling PyTorch DataParallel across 2 GPUs (cuda:0, cuda:1).
Using Standard Cross-Entropy Loss.

--- Epoch 1/4 ---


/tmp/ipykernel_22/2114198636.py:75: FutureWarning: `torch.cuda.amp.GradScaler(args...)` is deprecated. Please use `torch.amp.GradScaler('cuda', args...)` instead.
  scaler = torch.cuda.amp.GradScaler(enabled=USE_FP16)
/tmp/ipykernel_22/2114198636.py:94: FutureWarning: `torch.cuda.amp.autocast(args...)` is deprecated. Please use `torch.amp.autocast('cuda', args...)` instead.
  with torch.cuda.amp.autocast(enabled=USE_FP16):
/tmp/ipykernel_22/2114198636.py:103: UserWarning: Detected call of `lr_scheduler.step()` before `optimizer.step()`. In PyTorch 1.1.0 and later, you should call them in the opposite order: `optimizer.step()` before `lr_scheduler.step()`.  Failure to do this will result in PyTorch skipping the first value of the learning rate schedule. See more details at https://pytorch.org/docs/stable/optim.html#how-to-adjust-learning-rate
  scheduler.step()


...

  Step  100/ 784 | Avg Loss: 0.0746 | LR: 4.14e-06
  Step  200/ 784 | Avg Loss: 0.0705 | LR: 3.06e-06
  Step  300/ 784 | Avg Loss: 0.0702 | LR: 2.12e-06
  Step  400/ 784 | Avg Loss: 0.0704 | LR: 1.35e-06
  Step  500/ 784 | Avg Loss: 0.0700 | LR: 7.43e-07
  Step  600/ 784 | Avg Loss: 0.0680 | LR: 3.13e-07
  Step  700/ 784 | Avg Loss: 0.0677 | LR: 6.55e-08
  Step  784/ 784 | Avg Loss: 0.0678 | LR: 0.00e+00


/tmp/ipykernel_22/2114198636.py:119: FutureWarning: `torch.cuda.amp.autocast(args...)` is deprecated. Please use `torch.amp.autocast('cuda', args...)` instead.
  with torch.cuda.amp.autocast(enabled=USE_FP16):


  --> Epoch 4 Validation: Macro-F1 = 87.53% | Accuracy = 92.18%
  Validation score did not improve. Early stopping counter: 2/2
  Early stopping triggered at Epoch 4.

Loading best weights for final test evaluation: /kaggle/working/models/pubmed_200k_rct/PubMedBERT-base/best_model.pt


/tmp/ipykernel_22/144664855.py:59: FutureWarning: `torch.cuda.amp.autocast(args...)` is deprecated. Please use `torch.amp.autocast('cuda', args...)` instead.
  with torch.cuda.amp.autocast(enabled=USE_FP16):



EVALUATION RESULTS: pubmed_200k_rct_PubMedBERT-base
  Accuracy:      92.13%
  Macro-F1:      87.26%
  Micro-F1:      92.13%
  Transition Consistency: 98.28%
  Per-Class F1 Scores:
    BACKGROUND  : Precision=73.32% | Recall=81.41% | F1=77.15%
    OBJECTIVE   : Precision=79.90% | Recall=67.06% | F1=72.92%
    METHODS     : Precision=95.31% | Recall=96.69% | F1=95.99%
    RESULTS     : Precision=95.64% | Recall=95.40% | F1=95.52%
    CONCLUSION  : Precision=95.00% | Recall=94.42% | F1=94.71%
  Artifacts saved to: /kaggle/working/models/pubmed_200k_rct/PubMedBERT-base


### Step 12: Master Benchmark Comparison Table
Summarize all strictly computed empirical results into a publication-ready comparative table.

In [12]:
print("=" * 80)
print("MASTER COMPARATIVE BENCHMARK TABLE")
print("=" * 80)

summary_rows = []
for res in all_experiment_results:
    row = {
        "Run Name": res["run_name"],
        "Accuracy (%)": round(res["accuracy"] * 100, 2),
        "Macro-F1 (%)": round(res["macro_f1"] * 100, 2),
        "Micro-F1 (%)": round(res["micro_f1"] * 100, 2),
        "Transition Consistency (%)": round(res["sequential_transition_consistency_pct"], 2)
    }
    for c in TARGET_CLASSES:
        row[f"F1: {c}"] = round(res["per_class"][c]["f1"] * 100, 2)
    summary_rows.append(row)

df_summary = pd.DataFrame(summary_rows)
print(df_summary.to_string(index=False))

summary_csv_path = os.path.join(MODELS_OUTPUT_DIR, "encoder_models_comparison_table.csv")
df_summary.to_csv(summary_csv_path, index=False)
print(f"\nComparison table saved to: {summary_csv_path}")

MASTER COMPARATIVE BENCHMARK TABLE
                             Run Name  Accuracy (%)  Macro-F1 (%)  Micro-F1 (%)  Transition Consistency (%)  F1: BACKGROUND  F1: OBJECTIVE  F1: METHODS  F1: RESULTS  F1: CONCLUSION
 medical_abstracts_5k_PubMedBERT-base         83.66         83.09         83.66                       95.47           85.41          76.81        81.19        85.59           86.47
medical_abstracts_5k_BioLinkBERT-base         84.03         83.43         84.03                       95.74           85.61          77.38        81.44        86.00           86.72
       pubmed_20k_rct_PubMedBERT-base         92.34         87.85         92.34                       96.89           81.63          69.69        96.43        95.89           95.61
      pubmed_20k_rct_BioLinkBERT-base         92.52         88.06         92.52                       96.85           81.59          70.16        96.53        96.05           95.98
      pubmed_200k_rct_PubMedBERT-base         92.13         

### Step 13: Incremental Artifact Packaging & GitHub Push
Push evaluation metrics, confusion matrix figures, and training summaries to the user's connected GitHub repository via Kaggle Secrets (`GH_TOKEN`, `GH_REPO_URL`).
Ensures zero token exposure in logs and checks every file size against the GitHub 100 MB hard limit.

In [13]:
def push_results_to_github():
    print("=" * 80)
    print("GITHUB PUSH AUTOMATION")
    print("=" * 80)
    
    # 1. Fetch credentials from Kaggle Secrets
    try:
        from kaggle_secrets import UserSecretsClient
        user_secrets = UserSecretsClient()
        gh_token = user_secrets.get_secret("GH_TOKEN")
        gh_repo_url = user_secrets.get_secret("GH_REPO_URL")
    except Exception as e:
        print(f"Kaggle Secrets not configured or unavailable: {e}")
        print("Skipping Git push. All files remain safely saved in /kaggle/working/models/.")
        return
        
    if not gh_token or not gh_repo_url:
        print("GH_TOKEN or GH_REPO_URL missing in Kaggle Secrets. Skipping Git push.")
        return
        
    # 2. Check file sizes (respect GitHub 100 MB limit)
    git_dir = os.path.join(WORKING_DIR, "git_sync_repo")
    os.makedirs(git_dir, exist_ok=True)
    
    clean_url = gh_repo_url.replace("https://", "").replace("http://", "")
    auth_repo_url = f"https://{gh_token}@{clean_url}"
    
    # 3. Clone or init
    print("Preparing Git repository...")
    repo_name = clean_url.split("/")[-1].replace(".git", "")
    clone_path = os.path.join(git_dir, repo_name)
    
    if not os.path.exists(clone_path):
        clone_cmd = f"git clone https://x-access-token:{gh_token}@{clean_url} {clone_path}"
        ret = subprocess.run(clone_cmd, shell=True, capture_output=True, text=True)
        if ret.returncode != 0:
            print(f"Git clone error: {ret.stderr}")
            return
            
    # Copy results into repo
    dest_results = os.path.join(clone_path, "results", "encoder_models")
    os.makedirs(dest_results, exist_ok=True)
    
    skipped_large_files = []
    copied_files = []
    
    for root, _, files in os.walk(MODELS_OUTPUT_DIR):
        for f in files:
            src_f = os.path.join(root, f)
            size_mb = os.path.getsize(src_f) / (1024 * 1024)
            rel_path = os.path.relpath(src_f, MODELS_OUTPUT_DIR)
            dst_f = os.path.join(dest_results, rel_path)
            os.makedirs(os.path.dirname(dst_f), exist_ok=True)
            
            # Enforce 99 MB threshold
            if size_mb >= 99.0:
                skipped_large_files.append((f, size_mb))
            else:
                import shutil
                shutil.copy2(src_f, dst_f)
                copied_files.append((f, size_mb))
                
    if skipped_large_files:
        print("\nFiles skipped from Git push (exceeded 99 MB GitHub limit):")
        for f, sz in skipped_large_files:
            print(f"  - {f} ({sz:.2f} MB)")
            
    print(f"\nCopied {len(copied_files)} files to repository ({dest_results}).")
    
    # Git add, commit, push
    orig_cwd = os.getcwd()
    os.chdir(clone_path)
    try:
        subprocess.run("git config user.name 'Thesis Kaggle Runner'", shell=True)
        subprocess.run("git config user.email 'thesis-bot@kaggle.internal'", shell=True)
        subprocess.run("git add results/encoder_models/", shell=True)
        
        commit_msg = f"[Experiment] Biomedical Encoders Trained: {len(all_experiment_results)} runs"
        subprocess.run(f'git commit -m "{commit_msg}"', shell=True)
        
        push_res = subprocess.run("git push origin main", shell=True, capture_output=True, text=True)
        if push_res.returncode == 0:
            print("Successfully pushed results to GitHub repository!")
            # Verification: print git log
            log_res = subprocess.run("git log -1 --oneline", shell=True, capture_output=True, text=True)
            print(f"Verified latest commit: {log_res.stdout.strip()}")
        else:
            print(f"Git push returned non-zero code. Error output: {push_res.stderr}")
    finally:
        os.chdir(orig_cwd)

push_results_to_github()

GITHUB PUSH AUTOMATION
Kaggle Secrets not configured or unavailable: Unexpected response from the service. Response: {'errors': ['No user secrets exist for kernel id 138533083 and label GH_TOKEN.'], 'error': {'code': 5}, 'wasSuccessful': False}.
Skipping Git push. All files remain safely saved in /kaggle/working/models/.
